In [202]:
import os
import sys
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, parent_dir)

from Frame import Frame
import Utils as Utils
import numpy as np
import matplotlib.pyplot as plt
import Plotters
from plyfile import PlyData

import pickle
import numpy as np

import matplotlib.pyplot as plt
import Utils
%matplotlib qt


path = 'C:/Users/Roni/Documents/gs_input/frames_model.pkl'


# dict_path = 'D:/Documents/data_for_gs/fly_gray/dict/frames_model.pkl'








path_output = 'I:/My Drive/Research/gaussian_splatting/gaussian_splatting_output/'
# path_output = 'D:/Documents/gaussian_model_output/'

# dict_path  = 'I:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/mov7_2024_11_12_darkan/frames_model.pkl'
# image_path = 'I:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/mov7_2024_11_12_darkan/'
model_name = 'fly_to_bee'
file_name = 'bee_model_dense_10000'

model_name = 'fly_to_fly'
file_name = 'fly_model'

dict_path  = 'I:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/mov30_2024_11_12_darkan/frames_model.pkl'
image_path = 'I:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/mov30_2024_11_12_darkan/'
# model_name = 'only_fly'
# file_name = 'fly_model'

# model_name = 'model_8_4_25_deform_rec'
# file_name = 'model_rotation_lr_center0.07_densify_grad_threshold_0.00035'

# model_name = 'model_run'
# dict_path = 'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/mov7_2024_11_12_darkan/frames_model.pkl'
# image_path =  'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/mov7_2024_11_12_darkan/'
# model_name = '3dgs_bee'
# file_name = 'bee_try'


path_angles = f'{path_output}/{model_name}/{file_name}_angles.pkl'
path_results = f'{path_output}/{model_name}/{file_name}_angles.pkl'


# download model_run localy
path = f'D:/Documents/gaussian_model_output/{model_name}/{file_name}.pkl'
if os.path.exists(f'{path}'):
    with open(path, 'rb') as handle:
        output_angles_weights = pickle.load(handle)

iteration = 1200

frame0 = 1430
frame_end = 1431
weight_flag = False

with open(dict_path,'rb') as f:
    frames = pickle.load(f)

vertices_list = []
image_list = []
weights_list = []
gaussian_list = []
idx_parts = []
xyz_rotated = []
for frame in range(frame0,frame_end):
    ew_to_lab = frames[frame][1][list(frames[frame][1].keys())[0]]['ew_to_lab']
    input_dir = f'{path_output}/{model_name}/{file_name}'
    input_file = f'{path_output}/{model_name}/{frame}/{file_name}/point_cloud/iteration_{iteration}/point_cloud.ply'
    vertices = PlyData.read(input_file)["vertex"]
    xyz = np.column_stack((vertices['x'],vertices['y'],vertices['z']))
    vertices_list.append(xyz)
    xyz_rotated.append((ew_to_lab @ xyz.T).T)

    frames_per_cam = [Frame(image_path,frame,cam, frames_dict = frames)  for cam in range(4)]
    image_list.append(frames_per_cam)
    if os.path.exists(f'{input_dir}_results.pkl'):
        with open(f'{input_dir}_results.pkl', 'rb') as handle:
            output_angles_weights = pickle.load(handle)
        weights_list.append(output_angles_weights['weights'])
        weight_flag = True
        idx_parts.append([np.sum(output_angles_weights['weights'][frame - frame0][iteration][:,idx:idx + 3],axis = 1) == 1 for idx in range(0,9,3)])
        color_list = ['lime','crimson','dodgerblue']
        color_list_2d = ['lime','crimson','dodgerblue']





# frames_per_cam = [Frame(image_path,frame,cam_num, frames_dict = frames) for cam_num in range(4)]


In [251]:
import numpy as np
from scipy.linalg import svd
import pandas as pd
import plotly.graph_objects as go

frame_idx = idx_parts[frame - frame0]
body_xyz = xyz_rotated[frame - frame0][frame_idx[0],:]
r_wing_xyz = xyz_rotated[frame - frame0][frame_idx[1],:]
l_wing_xyz = xyz_rotated[frame - frame0][frame_idx[2],:]

def get_principle_axes(frame_xyz):
    body_cm = np.mean(frame_xyz,axis = 0)
    body_centered = frame_xyz - body_cm
    U, S, Vt = svd(body_centered, full_matrices=False)
    return Vt

def get_axis_orientation(axis,points_from,points_to):
    direction = (np.mean(points_to,axis = 0) - points_from)/np.linalg.norm(np.mean(points_to,axis = 0) - points_from)
    return -axis if np.dot(direction,axis) < 0 else axis


def reorient_axis(points,direction,percent = 0.2):
    projected_on_body = np.dot(points,direction)
    min_points = min(projected_on_body)
    max_points = max(projected_on_body)
    perc_of_body_length = (max_points - min_points)*percent
    bottom = points[(projected_on_body  < (min_points + perc_of_body_length)),:]
    top = points[(projected_on_body  > (max_points - perc_of_body_length)),:]
    x_ax = np.mean(top,axis = 0) - np.mean(bottom,axis = 0)
    return x_ax/np.linalg.norm(x_ax),bottom,top
    


   
body_cm = np.mean(body_xyz,axis = 0)
xbody = get_principle_axes(body_xyz)[0]
xbody = get_axis_orientation(xbody,[[0,0,0]],[[0,0,1]])

rwing_axes = get_principle_axes(r_wing_xyz)
lwing_axes = get_principle_axes(l_wing_xyz)

span_rw = get_axis_orientation(rwing_axes[0],body_cm,r_wing_xyz)
span_lw = get_axis_orientation(rwing_axes[0],body_cm,l_wing_xyz)

chord_rw = get_axis_orientation(rwing_axes[1],body_cm,r_wing_xyz)
chord_lw = get_axis_orientation(rwing_axes[1],body_cm,l_wing_xyz)

xbody,bottom,top = reorient_axis(body_xyz,xbody,percent = 0.2)



In [ ]:


data = projected_on_span
bins = 10
slices = np.linspace(0, 100, bins+1, True).astype(np.int)
counts = np.diff(slices)

mean = np.add.reduceat(data, slices[:-1]) / counts
mean

C:\Users\Roni\AppData\Local\Temp\ipykernel_29096\1998711944.py:5: DeprecationWarning:

`np.int` is a deprecated alias for the builtin `int`. To silence this warning, use `int` by itself. Doing this will not modify any behavior and is safe. When replacing `np.int`, you may wish to use e.g. `np.int64` or `np.int32` to specify the precision. If you wish to review your current use, check the release note link for additional information.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations



array([-0.0110284 , -0.011311  , -0.01109657, -0.01106716, -0.01118016,
       -0.01105592, -0.01105224, -0.01108058, -0.01095385, -1.56181881])

In [290]:
(min(projected_on_span)- max(projected_on_span))/100

-2.4839378963333916e-05

In [297]:
diff


2.4839378963333916e-05

In [ ]:
min_val = np.min(projected_on_span)
max_val = np.max(projected_on_span)
# Define bin edges
bin_edges = np.arange(min_val, max_val + diff, diff)

# Digitize: assign each value to a bin
bin_indices = np.digitize(projected_on_span, bins=bin_edges)


array([-0.01103944, -0.0110586 , -0.01105467, -0.01104781, -0.01105128,
       -0.01105533, -0.01104392, -0.011042  , -0.01105978, -0.01104994,
       -0.0110505 , -0.01105905, -0.01105962, -0.01106136, -0.01104525,
       -0.01105607, -0.01105909])

In [313]:
bin_indices

array([70, 71, 70, ..., 19, 90,  8], dtype=int64)

In [326]:
# fit_plane


projected_on_span = np.dot(r_wing_xyz,span_rw)
projected_on_chord = np.dot(r_wing_xyz,chord_rw)

diff = (max(projected_on_span) - min(projected_on_span))/100
bin_edges = np.arange(np.min(projected_on_span), np.max(projected_on_span) + diff, diff)
bin_indices = np.digitize(projected_on_span, bins=bin_edges)
real_indices = np.array(range(len(projected_on_chord)))
coord = []
for idx in bin_indices:
    max_of_bin = np.argmax(projected_on_chord[bin_indices == idx])
    real_idx = real_indices[bin_indices == idx][max_of_bin]
    coord.append(r_wing_xyz[real_idx,:])

le_points = np.vstack(coord)

fig = go.Figure()
Plotters.scatter3d(fig,r_wing_xyz,'red',3,'wing',show_colorbar = False)
Plotters.scatter3d(fig,l_wing_xyz,'blue',3,'wing',show_colorbar = False)
Plotters.scatter3d(fig,body_xyz,'green',3,'wing',show_colorbar = False)
Plotters.scatter3d(fig,le_points,'black',3,'wing',show_colorbar = False)


Plotters.scatter3d(fig, np.vstack((np.mean(r_wing_xyz,axis = 0),np.mean(r_wing_xyz,axis = 0) + r_xwing*2/1000)),'black',5,'x',mode = 'markers+lines') 
Plotters.scatter3d(fig, np.vstack((np.mean(l_wing_xyz,axis = 0),np.mean(l_wing_xyz,axis = 0) + l_xwing*2/1000)),'black',5,'x',mode = 'markers+lines') 
Plotters.scatter3d(fig, np.vstack((np.mean(body_xyz,axis = 0),np.mean(body_xyz,axis = 0) + xbody*2/1000)),'black',5,'x',mode = 'markers+lines') 
fig.show()




In [327]:
plt.plot(coord)

In [249]:
fig = go.Figure()
Plotters.scatter3d(fig,r_wing_xyz,'red',3,'wing',show_colorbar = False)
Plotters.scatter3d(fig,l_wing_xyz,'blue',3,'wing',show_colorbar = False)
Plotters.scatter3d(fig,body_xyz,'green',3,'wing',show_colorbar = False)


Plotters.scatter3d(fig, np.vstack((np.mean(r_wing_xyz,axis = 0),np.mean(r_wing_xyz,axis = 0) + r_xwing*2/1000)),'black',5,'x',mode = 'markers+lines') 
Plotters.scatter3d(fig, np.vstack((np.mean(l_wing_xyz,axis = 0),np.mean(l_wing_xyz,axis = 0) + l_xwing*2/1000)),'black',5,'x',mode = 'markers+lines') 
Plotters.scatter3d(fig, np.vstack((np.mean(body_xyz,axis = 0),np.mean(body_xyz,axis = 0) + xbody*2/1000)),'black',5,'x',mode = 'markers+lines') 
fig.show()

In [233]:
r_xwing_orient,bottom,top = reorient_axis(r_wing_xyz,r_xwing,percent = 0.1)
arrow = np.vstack((np.mean(bottom,axis = 0),np.mean(bottom,axis = 0) + r_xwing*3/1000))

fig = go.Figure()
Plotters.scatter3d(fig, r_wing_xyz,'green',3,'x') 
Plotters.scatter3d(fig, np.vstack((np.mean(bottom,axis = 0),np.mean(bottom,axis = 0) + r_xwing*4/1000)),'black',5,'x',mode = 'markers+lines') 
Plotters.scatter3d(fig, bottom,'red',3,'x') 
Plotters.scatter3d(fig, top,'red',3,'x') 
Plotters.scatter3d(fig, arrow,'blue',3,'x',mode = 'markers+lines') 
fig.show()